# Task 4 — YOLOv8 5-fold + masked pool (Kaggle판, fold 분담 학습용)

**목적**: Colab판(`task4_yolov8_5fold_masked_colab`)과 **완전히 동일한 데이터·설정**으로, Kaggle GPU에서
지정한 fold만 학습해 체크포인트를 분담 생산한다. **fold0는 Colab에서 진행 중**이므로 이 노트북은
기본적으로 fold1~4를 담당한다 (학습할 fold는 14번 셀 `FOLDS_THIS_SESSION`으로 지정).

| 항목 | 내용 |
|---|---|
| 데이터/분할/loss | Colab판과 동일: task2-masked 데이터(원본+pool2+masked, `syn_00505/00102` 제외, corrections 구버전 스냅샷) + `fold_split_masked.json` 고정 분할 + YOLOv8m, imgsz 960, cls gain **1.5** |
| 체크포인트 | `{tag}_fold{i}_best.pt` — **Colab판과 파일명 체계 동일** → 나중에 한 폴더에 모으면 앙상블 노트북에서 그대로 사용 |
| 세션 전략 | 세션당 fold 1개 권장 (12h 한도). 완료 fold는 `backup_dir`에 best.pt가 있으면 자동 스킵 |

**실행 전제 (Input 연결 4가지)**
- competition 데이터(`train_images`, `train_annotations`)
- 합성 pool(`task2_synthesized`) + masked pool(`dataset-74-masked`)
- **`fold_split_masked.json`** — Drive에 있는 파일을 그대로 **Kaggle Dataset으로 업로드**해 Input으로 연결
  (또는 이 json을 output에 포함한 task2-masked 커밋의 Output을 Input으로 연결. 파일명만 유지되면 자동 검색됨)

**운영 방법**
1. 14번 셀에서 `FOLDS_THIS_SESSION = [1]` 처럼 이번 세션에 학습할 fold를 지정
2. **Save Version(Save & Run All)** 으로 배치 실행 (인터랙티브 장시간 실행은 연결 끊김 위험)
3. 커밋 완료 후 Output의 `outputs/*_best.pt` 확인 → 다음 세션에서 다음 fold 지정 후 반복
4. 이미 완료한 fold의 체크포인트를 Input으로 연결하고 아래 선택 셀로 복사해두면 실수로 같은 fold를
   다시 돌려도 자동 스킵됨

**Kaggle Notebook: GPU on / Internet on** (ultralytics 설치 + yolov8m.pt 다운로드에 필요)


In [ ]:
# [1. 환경 준비] ultralytics(YOLOv8 학습 + COCO->YOLO 변환 유틸 포함)
#  ⚠ pip 설치는 세션이 재시작되면 사라지므로 매 세션 이 셀부터 실행해야 합니다.
!pip install -q "ultralytics>=8.3"

# 설치 검증: -q 옵션이 에러 로그를 가리는 경우 대비
import importlib
importlib.import_module('ultralytics')
print('설치 확인 OK: ultralytics')


In [ ]:
# [2. 저장소 준비] 팀 저장소 main 브랜치를 clone하고 RF_DETR_split_ver를 import 경로에 추가합니다.
#  ⚠ 저장소 파일/함수는 수정하지 않고 그대로 재사용합니다. (YOLO 관련 로직은 이 노트북에서 로컬로 정의)
import os, sys, re, glob, json, shutil
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
from PIL import Image

REPO_URL = 'https://github.com/kim-tae-yoon-0718/ai12-team01.git'
REPO_DIR = '/kaggle/working/ai12-team01'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
sys.path.insert(0, os.path.join(REPO_DIR, 'RF_DETR_split_ver'))

# ── 저장소에서 그대로 재사용하는 함수 (데이터 전처리 - 모델과 무관한 순수 로직) ──
from dataset import (
    load_raw_annotations, # 박스당 1개로 흩어진 원본 annotation을 파일명 기준 병합 로드
    apply_corrections,    # corrections 기반 bbox 수정
    # make_folds는 일부러 import하지 않음 - fold 분할은 fold_split_masked.json 로드로 고정 (재계산 금지)
    cache_images,         # 이미지 로컬 캐시
    write_fold_dirs,      # fold별 {train,valid}에 COCO json + 이미지 배치
    save_label_map,       # label_map.json 저장
    find_input_dir,       # /kaggle/input 아래에서 폴더 "이름"으로 재귀 탐색 (find_data_root의 Kaggle용 짝)
)

# ── RF_DETR_split_ver 공용 모듈 - task2-masked/task4-colab과 반복되던 pool 병합/74종 매핑/
#    fold 로드/YOLO 학습 로직을 모듈화한 것 (상세는 각 .py docstring 참고) ──
from pools import (
    load_pool_annotations, merge_pool,        # 합성 pool 로드(원본 id 공간 복원) + 병합
    merge_masked_pool,                          # masked pool 'msk_' 리네임 병합 (74종 매핑/이미지 존재 검증 포함)
    apply_exclusions, build_cat2label_74,       # 불량 이미지 제외 / 74종 라벨 매핑
)
from folds import (
    load_fold_split, assert_no_group_leak,      # 고정 분할 로드 + 그룹 누수 검증
    summarize_fold_distribution, print_fold_warnings,
)
from yolo import (
    get_yolo_model, build_yolo_fold,             # YOLOv8 모델 로드 / COCO->YOLO 변환
    train_fold_yolo, report_fold_result_yolo,    # fold 학습 + valid mAP 평가
    run_folds_yolo,                              # fold 목록을 받아 학습+리포팅 반복 (Colab판의 fold 지정 버전)
)
from ultralytics import YOLO
from ultralytics.data.converter import convert_coco   # COCO json -> YOLO txt 공식 변환 유틸

print('저장소 clone 완료:', REPO_DIR)


In [ ]:
# [3. corrections] task2-masked(Kaggle) 실험과 완전히 동일한 스냅샷을 사용합니다.
#  ⚠ 저장소 canonical corrections.json과 일부 다른 "구버전" 스냅샷입니다
#     (fix_category "3444"->3351 및 "791"->31863 포함, add_boxes 1건 category 상이).
#     fold-matched WBF 앙상블 파트너인 task2-masked RF-DETR 체크포인트와 학습 라벨 조건을
#     완전히 일치시키기 위해 의도적으로 이 스냅샷을 고정합니다.
#  ⚠ task2-masked PR(#13)에서 저장소 corrections.json 자체를 이 스냅샷으로 맞췄으므로,
#     이제 하드코딩 없이 파일을 직접 참조합니다.
CORRECTIONS_PATH = os.path.join(REPO_DIR, 'RF_DETR_split_ver', 'corrections.json')
print('corrections.json 존재:', os.path.exists(CORRECTIONS_PATH))


In [ ]:
# [4. 입력 경로 자동 탐색] (Kaggle)
#  /kaggle/input 아래에서 폴더 "이름"으로 찾기 때문에 dataset 슬러그를 몰라도 동작합니다
#  (dataset.find_input_dir - 저장소 공용 함수).
TRAIN_IMG_DIR = find_input_dir('train_images')       # 원본 train 이미지 (png)
TRAIN_ANN_DIR = find_input_dir('train_annotations')  # 원본 annotation (박스당 json 1개)

POOL_DIR      = find_input_dir('task2_synthesized')   # 합성 pool 루트 (images/, annotations/)
POOL_IMG_DIR  = os.path.join(POOL_DIR, 'images') if POOL_DIR else None
POOL_ANN_PATH = ((glob.glob(os.path.join(POOL_DIR, 'annotations', '*.json')) or [None])[0]
                 if POOL_DIR else None)

MASKED_DIR      = find_input_dir('dataset-74-masked')  # masked pool 루트 (images/, annotations/)
MASKED_IMG_DIR  = os.path.join(MASKED_DIR, 'images') if MASKED_DIR else None
MASKED_ANN_PATH = ((glob.glob(os.path.join(MASKED_DIR, 'annotations', '*.json')) or [None])[0]
                   if MASKED_DIR else None)

# 고정 fold 분할: 파일명으로 /kaggle/input 전체 재귀 검색 (Dataset이든 이전 커밋 Output이든 무관)
_hits = glob.glob('/kaggle/input/**/fold_split_masked.json', recursive=True)
assert _hits, ("fold_split_masked.json을 /kaggle/input 아래에서 못 찾음 - "
               "json을 담은 Dataset(또는 커밋 Output)이 Input으로 연결됐는지 확인.\n"
               f"현재 연결된 Input 폴더: {sorted(os.listdir('/kaggle/input'))}")
if len(_hits) > 1:
    print(f'fold_split_masked.json 후보 {len(_hits)}개 -> 첫 번째 사용:\n  ' + '\n  '.join(_hits))
FOLD_SPLIT_JSON = _hits[0]

paths = {'TRAIN_IMG_DIR': TRAIN_IMG_DIR, 'TRAIN_ANN_DIR': TRAIN_ANN_DIR,
         'POOL_IMG_DIR': POOL_IMG_DIR, 'POOL_ANN_PATH': POOL_ANN_PATH,
         'MASKED_IMG_DIR': MASKED_IMG_DIR, 'MASKED_ANN_PATH': MASKED_ANN_PATH,
         'FOLD_SPLIT_JSON': FOLD_SPLIT_JSON}
for k, v in paths.items():
    print(f'{k:16s}:', v)
assert all(paths.values()), "자동 탐색에 실패한 경로가 있습니다. 위 상수에 직접 경로를 지정하세요."


In [ ]:
# [5. 실험 설정] Colab판과 완전히 동일한 하이퍼파라미터 - 경로만 Kaggle 기준입니다.
#  tag도 동일하게 유지: fold별 체크포인트 파일명이 Colab 산출물과 같은 체계가 되어
#  나중에 한 폴더에 모아 앙상블 노트북에서 그대로 사용할 수 있습니다.
TASK_ID = 4
TAG = 'yolov8m_task4_syn74_masked'

config = {
    'data': {
        'n_splits': 5,
        'seed': 42,          # (참고) fold 분할은 fold_split_masked.json 로드로 고정 - seed는 분할에 관여하지 않음
        'dataset_dir': '/kaggle/tmp/dataset',             # COCO 포맷 fold 디렉토리
        'cache_dir': '/kaggle/tmp/img_cache',
        'yolo_dataset_dir': '/kaggle/tmp/yolo_dataset',   # YOLO 포맷(images/labels) fold 디렉토리
    },
    'model': {
        'variant': 'medium',   # yolov8m
        'tag': TAG,
    },
    'train': {
        'epochs': 50,
        'imgsz': 960,
        'batch': 8,             # Kaggle 16GB GPU 기준 (OOM 시 4로 축소)
        'patience': 10,         # val 개선이 10ep 없으면 조기 종료
        'seed': 42,
        'box_gain': 7.5,        # 기본값 유지
        'cls_gain': 1.5,        # 기본 0.5의 3배 - classification loss 패널티 강화 (Colab판과 동일)
        'dfl_gain': 1.5,        # 기본값 유지
    },
    'output': {
        'local_output_dir': '/kaggle/tmp/outputs',
        'backup_dir': '/kaggle/working/outputs',   # 체크포인트/학습곡선 (커밋 시 보존)
    },
}

for d in (config['data']['cache_dir'], config['data']['dataset_dir'],
          config['data']['yolo_dataset_dir'], config['output']['local_output_dir'],
          config['output']['backup_dir']):
    os.makedirs(d, exist_ok=True)

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU(!) - 가속기 설정 확인')
print('backup_dir(커밋 시 보존):', config['output']['backup_dir'])
print('loss gains:', {k: v for k, v in config['train'].items() if k.endswith('_gain')})


In [ ]:
# [6. 원본 train 로드 + annotation 수정] Task2와 완전히 동일한 절차입니다.
boxes_by_image, cats_by_image, img_meta, ids_by_image = load_raw_annotations(TRAIN_ANN_DIR)
print(f"수정 전: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")

boxes_by_image, cats_by_image = apply_corrections(
    boxes_by_image, cats_by_image, ids_by_image, CORRECTIONS_PATH)
print(f"수정 후: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")

train_cats = sorted({c for cs in cats_by_image.values() for c in cs})
print('train 클래스 수:', len(train_cats))


In [ ]:
# [7. pool2 병합] task2-masked와 완전히 동일한 로직입니다 (pools.load_pool_annotations/merge_pool).
#  ⚠ fold 구성을 task2-masked와 일치시켜야 하므로, 병합 로직은 그대로 유지합니다.
pool_boxes, pool_cats, pool_ids, pool_meta, pool_coco = load_pool_annotations(POOL_ANN_PATH)
print(f"pool2: 이미지 {len(pool_meta)}장 / 박스 {sum(len(v) for v in pool_boxes.values())}개"
      f" / 클래스 {len({c for cs in pool_cats.values() for c in cs})}종")

merge_pool(boxes_by_image, cats_by_image, ids_by_image, img_meta,
          pool_boxes, pool_cats, pool_ids, pool_meta, pool_name='pool2')


In [ ]:
# [7-1. masked pool 병합] task2-masked(Kaggle) 노트북과 동일한 로직입니다 (pools.merge_masked_pool).
#   - masked pool의 _annotations.coco.json은 categories id가 "원본 category_id 그대로"라 name 매핑 없이 사용
#   - 파일명이 train과 동일 체계(K코드+촬영조건)라 원본과 겹칠 수 있어, 전체 파일을 'msk_' 접두어로
#     리네임한 사본을 스테이징 폴더에 생성 (cache_images 덮어쓰기·corrections 오적용 방지)
#   - 같은 구성(위도/촬영조건 변형 + masked 버전)의 split 누수 방지 그룹화는 fold_split_masked.json에
#     이미 반영되어 있고, 9번 셀에서 그룹 누수 assert로 재검증합니다
MASKED_STAGE_DIR = '/kaggle/tmp/masked_renamed'
cats_74 = {int(c['name']) for c in pool_coco['categories'] if c['id'] != 0}
n_masked, masked_coco = merge_masked_pool(
    boxes_by_image, cats_by_image, ids_by_image, img_meta,
    MASKED_IMG_DIR, MASKED_ANN_PATH, MASKED_STAGE_DIR, cats_allowed=cats_74)


In [ ]:
# [7-2. 데이터 제외] task2-masked와 동일한 제외 목록을 반드시 그대로 적용합니다 (pools.apply_exclusions).
#  ⚠ fold_split_masked.json은 이 제외가 적용된 파일 집합 기준으로 만들어졌으므로,
#     목록이 다르면 9번 셀의 "분할-데이터 일치" assert에서 중단됩니다.
EXCLUDE_FILES = [
    'syn_00505.png',
    'syn_00102.png',
]
apply_exclusions(EXCLUDE_FILES, boxes_by_image, cats_by_image, ids_by_image, img_meta)


In [ ]:
# [8. 74종 라벨 매핑] Task2와 동일한 체계 (pools.build_cat2label_74): train 56종 -> 1~56,
#  test 전용 18종 -> 57~74. fold 구성을 Task2와 맞추려면 이 매핑도 완전히 동일해야 합니다.
cat2label, label2cat, all_cats = build_cat2label_74(pool_coco, train_cats, cats_by_image)


In [ ]:
# [9. 5-fold 분할 로드 + fold별 클래스 배분 점검] - 고정 분할 (재계산 없음)
#  StratifiedGroupKFold를 다시 돌리지 않고, task2-masked에서 export한 fold_split_masked.json을
#  folds.load_fold_split으로 로드합니다 (분할-데이터 불일치는 내부 assert로 검증). 이유:
#   1) fold-matched WBF: task2-masked RF-DETR 체크포인트와 fold별 valid 이미지 집합이 동일해야 함
#   2) 분할 재계산은 세션/계정 간 sklearn·numpy 버전 차이로 다른 분할이 나올 위험이 있음
file_names = sorted(boxes_by_image)
folds = load_fold_split(FOLD_SPLIT_JSON, file_names, config['data']['n_splits'])

# 그룹 누수 점검: 같은 그룹('msk_' 접두어 제거 기준 구성 코드)이 train/valid 양쪽에 있으면 안 됩니다.
assert_no_group_leak(folds, file_names)

fold_summary, valid_pivot = summarize_fold_distribution(folds, file_names, cats_by_image, cat2label)
display(fold_summary)

print_fold_warnings(fold_summary)

valid_pivot   # 라벨 x fold별 valid 박스 수 (셀 마지막 줄 -> 표로 표시. 클래스별 집계에서 재사용)


In [ ]:
# 이 셀 바로 위에 추가해서 확인
print("=== category mapping 확인 ===")
print(f"총 클래스 수: {len(cat2label)}")
print(f"\n{'category_id':>12} {'dl_idx':>10}")
print("-" * 25)
for name, cat_id in sorted(cat2label.items(), key=lambda x: x[1]):
    print(f"  {cat_id:>10} {name:>10}")

In [ ]:
# [10. COCO 포맷 fold 디렉토리 생성] Task2와 동일한 저장소 함수 조합입니다.
#  이 산출물(fold{i}/{train,valid}/_annotations.coco.json + 이미지)을 다음 셀에서 YOLO 포맷으로 변환합니다.
cache_images(TRAIN_IMG_DIR, config['data']['cache_dir'])
cache_images(POOL_IMG_DIR, config['data']['cache_dir'])
cache_images(MASKED_STAGE_DIR, config['data']['cache_dir'])   # masked 리네임 사본(msk_*.png)도 같은 캐시에 추가

write_fold_dirs(folds, file_names, boxes_by_image, cats_by_image, img_meta,
                cat2label, all_cats, config['data']['cache_dir'], config['data']['dataset_dir'])

# fold 디렉토리 복사가 끝나면 캐시는 더 필요 없으므로 삭제해 디스크를 확보합니다
#  (masked pool 추가로 데이터가 커져 fold 5개 사본 + 캐시가 공존하면 디스크가 부족할 수 있음)
shutil.rmtree(config['data']['cache_dir'], ignore_errors=True)
print('이미지 캐시 삭제 완료 (fold 디렉토리에 이미 복사됨)')

save_label_map(cat2label, label2cat, config['data']['dataset_dir'])
print('COCO 포맷 fold 디렉토리 생성 완료:', config['data']['dataset_dir'])


## 11. YOLO 포맷 변환

Ultralytics는 객체 검출 학습에 COCO json을 직접 쓰지 않고 `images/` + `labels/`(YOLO txt) 구조와
`data.yaml`을 요구한다 (확인 결과: 세그멘테이션/포즈가 아닌 detection 학습에 COCO json을 그대로 읽는
네이티브 경로는 없음). 다만 공식 변환 유틸 `ultralytics.data.converter.convert_coco()`가 이 변환을
대신해주므로 직접 파싱하지 않고 그대로 사용한다.

- `cls91to80=False`로 호출하면 `class_index = category_id - 1`을 그대로 쓴다 (COCO 80/91클래스
  리매핑을 켜지 않음 — 우리 데이터는 원래 COCO 80종이 아니므로 반드시 꺼야 함).
- 저장소 `build_coco()`가 넣는 더미 배경 카테고리(`id=0`, name='pill')에는 박스가 하나도 없으므로,
  별도로 제거하지 않아도 실제 74종이 정확히 YOLO class index `0~73`으로 매핑된다.
- `convert_coco()`는 라벨 txt만 생성하고 이미지는 복사하지 않으므로, 이미지는 심볼릭 링크로 연결해
  디스크 중복을 피한다.
- 변환된 txt 파일명은 COCO json의 `file_name`과 동일한 stem을 쓰므로, `images/`와 `labels/`를
  같은 파일명 규칙으로 나란히 두면 Ultralytics가 자동으로 짝을 찾는다.


In [ ]:
# [11. YOLO 포맷 변환 실행] (yolo.build_yolo_fold - COCO fold -> YOLO images/labels + data.yaml)
fold_yaml_paths = [
    build_yolo_fold(fi, config['data']['dataset_dir'], config['data']['yolo_dataset_dir'], label2cat)
    for fi in range(config['data']['n_splits'])
]
print('data.yaml 경로:')
for p in fold_yaml_paths:
    print(' ', p)


## 14. 지정 fold 학습 실행

**fold0는 Colab에서 학습 중**이므로 이 노트북은 `FOLDS_THIS_SESSION`에 지정한 fold만 학습합니다.

- 세션당 fold 1개 권장: `FOLDS_THIS_SESSION = [1]` → 커밋 → 다음 세션 `[2]` → ...
  (yolov8m 100ep 기준 fold 1개가 수 시간이므로, 2개를 넣으면 12h를 넘길 수 있습니다.
  Colab fold0 소요 시간을 보고 세션당 개수를 정하세요)
- `train_fold_yolo()`는 `backup_dir`에 해당 fold의 `*_best.pt`가 이미 있으면 자동 스킵합니다.
  이미 완료한 fold의 커밋 Output을 Input으로 추가하고 아래 선택 셀로 복사해두면,
  실수로 같은 fold를 지정해도 다시 학습하지 않습니다.
- 학습이 끝나면 fold별 valid mAP가 출력되고, `results.csv`/`results.png`(학습곡선)도 함께 백업됩니다.


In [ ]:
# (선택) 이어하기: 이전 커밋 output(outputs/*_best.pt)을 Input으로 추가했다면 backup_dir로 복사
#  -> train_fold_yolo()가 이미 학습된 fold를 자동으로 건너뜁니다.
# PREV_OUTPUT_DIR = '/kaggle/input/<이전-커밋-슬러그>/outputs'
# for p in glob.glob(os.path.join(PREV_OUTPUT_DIR, '*_best.pt')):
#     shutil.copy(p, config['output']['backup_dir'])
#     print('복사:', os.path.basename(p))


In [ ]:
# [14. 지정 fold 학습 + 리포팅 실행] (yolo.run_folds_yolo - fold_indices로 지정한 fold만 학습)
FOLDS_THIS_SESSION = [1]   # 이번 세션에 학습할 fold 번호 목록 (fold0는 Colab 담당)

assert all(0 <= fi < config['data']['n_splits'] for fi in FOLDS_THIS_SESSION), 'fold 번호 범위 확인'

run_result = run_folds_yolo(config, fold_yaml_paths, fold_indices=FOLDS_THIS_SESSION)
print('\nbackup_dir 내용:', sorted(os.listdir(config['output']['backup_dir'])))


## 산출물

- `/kaggle/working/outputs/yolov8m_task4_syn74_masked_fold{i}_best.pt` — 이번 세션에 학습한 fold 체크포인트
  (**Colab판과 동일한 파일명 체계** — Colab의 fold0 + Kaggle의 fold1~4를 한 폴더에 모으면
  fold-matched WBF 앙상블 노트북에서 그대로 사용 가능)
- `..._fold{i}_results.csv` / `_results.png` — fold별 학습 곡선 (Ultralytics 자동 생성)
- 이 노트북은 체크포인트 분담 생산이 목적이라 test 추론/WBF/CSV는 수행하지 않습니다
  (Colab판 또는 별도 앙상블 노트북에서 진행).

**다음 세션**: 이 커밋의 Output을 Input으로 추가 → 선택 셀에서 복사 → `FOLDS_THIS_SESSION`을 다음 fold로 변경 → Save & Run All
